# Retina Super-Resolution — SRGAN (Google Colab)
Grayscale retinal OCT images, 1-channel SRGAN with pixel + perceptual + adversarial loss.

In [ ]:
!pip install -q scikit-image

In [ ]:
from google.colab import drive
# Set force_remount=True if you get 'Transport endpoint is not connected'
drive.mount('/content/drive', force_remount=False)

In [ ]:
import os, glob, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.layers import (
    Add, BatchNormalization, Conv2D, Dense, Flatten,
    Input, LeakyReLU, PReLU, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.optimizers import Adam
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── Paths & Hyperparameters ──────────────────────────────────────────────
BASE_DRIVE = '/content/drive/MyDrive/Retina_SR/data'  # <-- set your actual path

HR_DIRS = [
    os.path.join(BASE_DRIVE, 'High_Res', 'group1'),
    os.path.join(BASE_DRIVE, 'High_Res', 'group2'),
    os.path.join(BASE_DRIVE, 'High_Res', 'group3'),
]
LR_DIRS = [
    os.path.join(BASE_DRIVE, 'Low_Res', 'group1'),
    os.path.join(BASE_DRIVE, 'Low_Res', 'group2'),
    os.path.join(BASE_DRIVE, 'Low_Res', 'group3'),
]

SCALE          = 4      # upscale factor: 4 or 2 (auto-checked below)

# Override HR size (None = use actual image size, 256 recommended for free Colab GPU)
HR_SIZE_OVERRIDE = 256
NUM_FILTERS    = 64
NUM_RES_BLOCKS = 16
BATCH_SIZE     = 4      # reduce to 2 if OOM
PRETRAIN_STEPS = 1000
SRGAN_STEPS    = 2000
LR_G           = 1e-4
LR_D           = 1e-4
D_UPDATE_FREQ  = 1      # D update steps per G update
W_PIXEL        = 1.0
W_PERCEPTUAL   = 0.006
W_ADV          = 0.001

SAVE_DIR = '/content/srgan_results'
os.makedirs(os.path.join(SAVE_DIR, 'checkpoints'), exist_ok=True)
print(f'Config ready — SCALE={SCALE}x, BATCH_SIZE={BATCH_SIZE}')

In [ ]:
# ── Collect image paths & match by filename ──────────────────────────────
IMG_EXTS = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff')

def collect_paths(dirs):
    """Return dict: filename -> full_path"""
    result = {}
    for d in dirs:
        for ext in IMG_EXTS:
            for p in sorted(glob.glob(os.path.join(d, ext))):
                result[os.path.basename(p)] = p
    return result

hr_dict = collect_paths(HR_DIRS)
lr_dict = collect_paths(LR_DIRS)
print(f'HR images found: {len(hr_dict)}')
print(f'LR images found: {len(lr_dict)}')

# Keep only filenames present in both HR and LR
common = sorted(set(hr_dict.keys()) & set(lr_dict.keys()))
print(f'Matched pairs:   {len(common)}')
assert len(common) > 0, 'No matching filenames between HR and LR folders!'

hr_paths = [hr_dict[f] for f in common]
lr_paths = [lr_dict[f] for f in common]

# Auto-detect sizes from first image pair
s_hr = np.array(Image.open(hr_paths[0]).convert('L'))
s_lr = np.array(Image.open(lr_paths[0]).convert('L'))
print(f'HR original shape: {s_hr.shape}')
print(f'LR original shape: {s_lr.shape}')

actual_scale = s_hr.shape[0] // s_lr.shape[0]
print(f'Detected scale: {actual_scale}x  (config: SCALE={SCALE})')
if actual_scale != SCALE:
    print(f'[WARNING] Set SCALE={actual_scale} in the config cell and re-run!')

# HR_SIZE: target HR image size
# LR_SIZE: Generator input size = HR_SIZE // SCALE  (output will match HR_SIZE)
HR_SIZE = HR_SIZE_OVERRIDE if HR_SIZE_OVERRIDE else s_hr.shape[0]
LR_SIZE = HR_SIZE // SCALE
print(f'HR_SIZE={HR_SIZE}, LR_SIZE={LR_SIZE}  (G input: {LR_SIZE} -> output: {HR_SIZE})')

# Train / Val split 80 / 20
n_total = len(hr_paths)
rng_idx = np.random.permutation(n_total)
n_train = int(n_total * 0.8)
tr, va = rng_idx[:n_train], rng_idx[n_train:]
hr_train = [hr_paths[i] for i in tr]
lr_train = [lr_paths[i] for i in tr]
hr_val   = [hr_paths[i] for i in va]
lr_val   = [lr_paths[i] for i in va]
print(f'Train: {len(hr_train)}, Val: {len(hr_val)}')

In [ ]:
# ── tf.data pipeline — full image, resize to fixed size ─────────────────
# If Drive disconnects: drive.mount('/content/drive', force_remount=True)

def load_pair(hr_path, lr_path):
    def _read(p, h, w):
        img = tf.io.read_file(p)
        img = tf.image.decode_image(img, channels=1, expand_animations=False)
        img = tf.cast(img, tf.float32) / 255.0
        img = tf.image.resize(img, [h, w])
        return img

    hr = _read(hr_path, HR_SIZE, HR_SIZE)
    lr = _read(lr_path, LR_SIZE, LR_SIZE)

    # Horizontal flip augmentation
    do_flip = tf.random.uniform(()) > 0.5
    hr = tf.cond(do_flip, lambda: tf.image.flip_left_right(hr), lambda: hr)
    lr = tf.cond(do_flip, lambda: tf.image.flip_left_right(lr), lambda: lr)
    return lr, hr


def make_ds(hr_list, lr_list, bs, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((hr_list, lr_list))
    if shuffle:
        ds = ds.shuffle(len(hr_list))
    return ds.map(load_pair, num_parallel_calls=tf.data.AUTOTUNE).batch(bs).prefetch(tf.data.AUTOTUNE)


train_ds = make_ds(hr_train, lr_train, BATCH_SIZE)
val_ds   = make_ds(hr_val,   lr_val,   BATCH_SIZE, shuffle=False)

print('Testing pipeline...')
for lr_b, hr_b in val_ds.take(1):
    print(f'LR batch: {lr_b.shape}, HR batch: {hr_b.shape}')
    n_show = min(4, lr_b.shape[0])
    fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
    for i in range(n_show):
        axes[0,i].imshow(lr_b[i,...,0], cmap='gray')
        axes[0,i].set_title(f'LR {lr_b.shape[1]}x{lr_b.shape[2]}')
        axes[0,i].axis('off')
        axes[1,i].imshow(hr_b[i,...,0], cmap='gray')
        axes[1,i].set_title(f'HR {hr_b.shape[1]}x{hr_b.shape[2]}')
        axes[1,i].axis('off')
    plt.suptitle('Data Samples (LR top / HR bottom)')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'data_sample.png'), dpi=120)
    plt.show()
print('Pipeline OK')

In [ ]:
# ── SRGAN Model (grayscale 1-channel) ────────────────────────────────────
def norm_m11(x):    return x * 2.0 - 1.0          # [0,1] -> [-1,1]
def denorm_m11(x):  return (x + 1.0) / 2.0        # [-1,1] -> [0,1]
def pix_shuffle(s): return lambda x: tf.nn.depth_to_space(x, s)


def upsample_block(x_in, nf):
    x = Conv2D(nf, 3, padding='same')(x_in)
    x = Lambda(pix_shuffle(2))(x)
    return PReLU(shared_axes=[1, 2])(x)


def res_block(x_in, nf, mom=0.8):
    x = Conv2D(nf, 3, padding='same')(x_in)
    x = BatchNormalization(momentum=mom)(x)
    x = PReLU(shared_axes=[1, 2])(x)
    x = Conv2D(nf, 3, padding='same')(x)
    x = BatchNormalization(momentum=mom)(x)
    return Add()([x_in, x])


def build_generator(nf=64, n_res=16, scale=4):
    x_in = Input(shape=(None, None, 1))   # grayscale 1-channel input
    x = Lambda(norm_m11)(x_in)
    x = Conv2D(nf, 9, padding='same')(x)
    x = skip = PReLU(shared_axes=[1, 2])(x)
    for _ in range(n_res):
        x = res_block(x, nf)
    x = Conv2D(nf, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([skip, x])
    for _ in range(int(math.log2(scale))):
        x = upsample_block(x, nf * 4)
    x = Conv2D(1, 9, padding='same', activation='tanh')(x)  # 1-channel output
    x = Lambda(denorm_m11)(x)
    return Model(x_in, x, name='generator')


def disc_block(x_in, nf, s=1, bn=True, mom=0.8):
    x = Conv2D(nf, 3, strides=s, padding='same')(x_in)
    if bn:
        x = BatchNormalization(momentum=mom)(x)
    return LeakyReLU(0.2)(x)


def build_discriminator(hr_size, nf=64):
    x_in = Input(shape=(hr_size, hr_size, 1))  # grayscale 1-channel
    x = Lambda(norm_m11)(x_in)
    x = disc_block(x, nf,     bn=False)
    x = disc_block(x, nf,     s=2)
    x = disc_block(x, nf*2)
    x = disc_block(x, nf*2,   s=2)
    x = disc_block(x, nf*4)
    x = disc_block(x, nf*4,   s=2)
    x = disc_block(x, nf*8)
    x = disc_block(x, nf*8,   s=2)
    x = Flatten()(x)
    x = Dense(1024)(x)
    x = LeakyReLU(0.2)(x)
    x = Dense(1, activation='sigmoid')(x)
    return Model(x_in, x, name='discriminator')


def build_vgg(layer_idx=20):
    """grayscale (1ch) -> replicate to 3ch -> VGG19 feature extraction"""
    vgg = VGG19(input_shape=(None, None, 3), include_top=False, weights='imagenet')
    vgg.trainable = False
    feat = Model(vgg.input, vgg.layers[layer_idx].output)
    inp = Input(shape=(None, None, 1))
    x = Lambda(lambda t: tf.repeat(t * 255.0, 3, axis=-1))(inp)
    return Model(inp, feat(x), name='vgg_extractor')


generator     = build_generator(NUM_FILTERS, NUM_RES_BLOCKS, SCALE)
discriminator = build_discriminator(HR_SIZE, NUM_FILTERS)
vgg_extractor = build_vgg(layer_idx=20)

print(f'Generator     params: {generator.count_params():,}')
print(f'Discriminator params: {discriminator.count_params():,}')
generator.summary(line_length=80)

In [ ]:
# ── Loss Functions & Optimizers ──────────────────────────────────────────
opt_g = Adam(LR_G, beta_1=0.9)
opt_d = Adam(LR_D, beta_1=0.9)
bce   = tf.keras.losses.BinaryCrossentropy()
mse   = tf.keras.losses.MeanSquaredError()


@tf.function
def l_pixel(hr, sr):
    return mse(hr, sr)


@tf.function
def l_perceptual(hr, sr):
    return mse(vgg_extractor(hr, training=False),
               vgg_extractor(sr, training=False))


@tf.function
def l_adv_g(sr):
    p = discriminator(sr, training=False)
    return bce(tf.ones_like(p), p)


@tf.function
def l_disc(hr, sr):
    r = discriminator(hr, training=True)
    f = discriminator(sr, training=True)
    return 0.5 * (bce(tf.ones_like(r), r) + bce(tf.zeros_like(f), f))


@tf.function
def pretrain_step(lr_b, hr_b):
    with tf.GradientTape() as t:
        sr   = generator(lr_b, training=True)
        loss = l_pixel(hr_b, sr)
    opt_g.apply_gradients(
        zip(t.gradient(loss, generator.trainable_variables),
            generator.trainable_variables))
    return loss


@tf.function
def srgan_g_step(lr_b, hr_b):
    with tf.GradientTape() as t:
        sr    = generator(lr_b, training=True)
        lp    = l_pixel(hr_b, sr)
        lperc = l_perceptual(hr_b, sr)
        ladv  = l_adv_g(sr)
        total = W_PIXEL*lp + W_PERCEPTUAL*lperc + W_ADV*ladv
    opt_g.apply_gradients(
        zip(t.gradient(total, generator.trainable_variables),
            generator.trainable_variables))
    return total, lp, lperc, ladv


@tf.function
def srgan_d_step(lr_b, hr_b):
    sr = generator(lr_b, training=False)
    with tf.GradientTape() as t:
        loss = l_disc(hr_b, sr)
    opt_d.apply_gradients(
        zip(t.gradient(loss, discriminator.trainable_variables),
            discriminator.trainable_variables))
    return loss

In [ ]:
# ── Phase 1: Generator Pre-training (MSE only) ───────────────────────────
print('=== Phase 1: Pre-training Generator with MSE ===')
pre_losses = []
it = iter(train_ds.repeat())

for step in range(1, PRETRAIN_STEPS + 1):
    lr_b, hr_b = next(it)
    loss = pretrain_step(lr_b, hr_b)
    pre_losses.append(float(loss))
    if step % 100 == 0 or step == 1:
        print(f'  [{step:4d}/{PRETRAIN_STEPS}]  MSE: {loss:.6f}')

plt.figure(figsize=(8, 3))
plt.plot(pre_losses)
plt.title('Pre-train Generator Loss (MSE)')
plt.xlabel('Step'); plt.ylabel('MSE')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pretrain_loss.png'), dpi=120)
plt.show()

generator.save_weights(os.path.join(SAVE_DIR, 'checkpoints', 'gen_pretrained.weights.h5'))
print('Saved pretrained weights.')

In [ ]:
# ── Phase 2: SRGAN Training (pixel + perceptual + adversarial) ───────────
print('=== Phase 2: SRGAN Training ===')
log = {k: [] for k in ('g_total', 'g_pix', 'g_perc', 'g_adv', 'd_loss')}
it = iter(train_ds.repeat())

for step in range(1, SRGAN_STEPS + 1):
    d_loss = 0.0
    for _ in range(D_UPDATE_FREQ):
        lr_b, hr_b = next(it)
        d_loss = float(srgan_d_step(lr_b, hr_b))

    lr_b, hr_b = next(it)
    g_tot, g_pix, g_perc, g_adv = srgan_g_step(lr_b, hr_b)

    for k, v in zip(('g_total', 'g_pix', 'g_perc', 'g_adv', 'd_loss'),
                    (g_tot, g_pix, g_perc, g_adv, d_loss)):
        log[k].append(float(v))

    if step % 200 == 0 or step == 1:
        print(f'  [{step:4d}/{SRGAN_STEPS}]  '
              f'G={g_tot:.4f}  pix={g_pix:.4f}  '
              f'perc={g_perc:.4f}  adv={g_adv:.4f}  D={d_loss:.4f}')

generator.save_weights(os.path.join(SAVE_DIR, 'checkpoints', 'gen_srgan.weights.h5'))
discriminator.save_weights(os.path.join(SAVE_DIR, 'checkpoints', 'disc_srgan.weights.h5'))
print('Saved SRGAN weights.')

In [ ]:
# ── Training Curves ──────────────────────────────────────────────────────
steps = range(1, len(log['g_total']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(steps, log['g_total'])
axes[0].set_title('G Total Loss'); axes[0].set_xlabel('Step')

axes[1].plot(steps, log['g_pix'],  label=f'Pixel x{W_PIXEL}')
axes[1].plot(steps, log['g_perc'], label=f'Perceptual x{W_PERCEPTUAL}')
axes[1].plot(steps, log['g_adv'],  label=f'Adversarial x{W_ADV}')
axes[1].set_title('G Loss Components'); axes[1].set_xlabel('Step'); axes[1].legend()

axes[2].plot(steps, log['d_loss'], color='orange')
axes[2].set_title('D Loss'); axes[2].set_xlabel('Step')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── Quantitative Evaluation (PSNR / SSIM) ────────────────────────────────
def evaluate(ds, n_batches=10):
    psnr_lr, psnr_sr, ssim_lr, ssim_sr = [], [], [], []
    for i, (lr_b, hr_b) in enumerate(ds):
        if i >= n_batches:
            break
        sr_b = np.clip(generator(lr_b, training=False).numpy(), 0, 1)
        for j in range(lr_b.shape[0]):
            hr   = hr_b[j, ..., 0].numpy()
            sr   = sr_b[j, ..., 0]
            lr_u = np.array(
                Image.fromarray((lr_b[j, ..., 0].numpy() * 255).astype(np.uint8))
                     .resize((HR_SIZE, HR_SIZE), Image.BICUBIC)
            ).astype(np.float32) / 255.0
            psnr_lr.append(calc_psnr(hr, lr_u, data_range=1.0))
            psnr_sr.append(calc_psnr(hr, sr,   data_range=1.0))
            ssim_lr.append(calc_ssim(hr, lr_u,  data_range=1.0))
            ssim_sr.append(calc_ssim(hr, sr,    data_range=1.0))
    return {
        'PSNR Bicubic': np.mean(psnr_lr),
        'PSNR SRGAN':   np.mean(psnr_sr),
        'SSIM Bicubic': np.mean(ssim_lr),
        'SSIM SRGAN':   np.mean(ssim_sr),
    }


metrics = evaluate(val_ds)
print('=== Validation Set Metrics ===')
for k, v in metrics.items():
    print(f'  {k:20s}: {v:.4f}')

In [ ]:
# ── Result Images (LR / SR / HR comparison) ──────────────────────────────
def show_results(ds, n=4, save_path=None):
    for lr_b, hr_b in ds.take(1):
        sr_b = np.clip(generator(lr_b, training=False).numpy(), 0, 1)
        lr_b = lr_b.numpy()
        hr_b = hr_b.numpy()
        break

    n = min(n, lr_b.shape[0])
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[None]

    for i in range(n):
        hr   = hr_b[i, ..., 0]
        sr   = sr_b[i, ..., 0]
        lr_u = np.array(
            Image.fromarray((lr_b[i, ..., 0] * 255).astype(np.uint8))
                 .resize((HR_SIZE, HR_SIZE), Image.BICUBIC)
        ).astype(np.float32) / 255.0

        p_lr = calc_psnr(hr, lr_u, data_range=1.0)
        p_sr = calc_psnr(hr, sr,   data_range=1.0)
        s_lr = calc_ssim(hr, lr_u,  data_range=1.0)
        s_sr = calc_ssim(hr, sr,    data_range=1.0)

        titles = [
            f'LR (Bicubic)\nPSNR={p_lr:.2f} SSIM={s_lr:.4f}',
            f'SR (SRGAN)\nPSNR={p_sr:.2f} SSIM={s_sr:.4f}',
            'HR (Ground Truth)',
        ]
        for ax, img, title in zip(axes[i], [lr_u, sr, hr], titles):
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
            ax.set_title(title, fontsize=9)
            ax.axis('off')

    plt.suptitle('LR (Bicubic)  |  SR (SRGAN)  |  HR (Ground Truth)', fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


show_results(val_ds, n=4,
             save_path=os.path.join(SAVE_DIR, 'result_comparison.png'))
print(f'Saved: {SAVE_DIR}/result_comparison.png')

In [ ]:
# ── GAN Stability Guide (adjust and re-run Phase 2 if needed) ────────────
#
# D loss -> 0  (D wins): lower LR_D or set D_UPDATE_FREQ = 0
# G adv loss explodes  : lower W_ADV = 0.0005
# No quality gain      : raise W_PERCEPTUAL = 0.01, increase PRETRAIN_STEPS
#
# LR decay schedule example:
# sched = tf.keras.optimizers.schedules.ExponentialDecay(
#     1e-4, decay_steps=500, decay_rate=0.5, staircase=True)
# opt_g = Adam(sched)
#
# Label smoothing (stabilizes D):
# real labels -> 0.9,  fake labels -> 0.1
print('GAN stability guide: see comments above')

In [ ]:
# ── Save Results to Google Drive ─────────────────────────────────────────
import shutil
dst = os.path.normpath(os.path.join(BASE_DRIVE, '..', 'srgan_results'))
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(SAVE_DIR, dst)
print(f'Saved to Drive: {dst}')
for f in sorted(os.listdir(dst)):
    print(' ', f)